In [1]:
from typing import Tuple
from enum import IntEnum

from brax import base
from brax.envs.base import PipelineEnv, State
from brax.io import mjcf
from etils import epath
import jax
from jax import numpy as jp
import mujoco 
from mujoco import mj_id2name, mj_name2id
from mujoco.mjx._src.support import contact_force # this import causes the warp warning due to some imports in that file idk why or if important


Failed to import warp: No module named 'warp'
Failed to import mujoco_warp: No module named 'warp'


In [ ]:
path = "/home/leo/assistive-autonomy-github/assistax/assistax/envs/assets/handover.xml"
mjmodel = mujoco.MjModel.from_xml_path(str(path))
sys = mjcf.load_model(mjmodel)

In [ ]:
panda1_left_finger_geom = mj_name2id(mjmodel, mujoco.mjtObj.mjOBJ_GEOM, "panda1_left_finger_pad")
panda1_right_finger_geom = mj_name2id(mjmodel, mujoco.mjtObj.mjOBJ_GEOM, "panda1_right_finger_pad")
panda2_left_finger_geom = mj_name2id(mjmodel, mujoco.mjtObj.mjOBJ_GEOM, "panda2_left_finger_pad")
panda2_right_finger_geom = mj_name2id(mjmodel, mujoco.mjtObj.mjOBJ_GEOM, "panda2_right_finger_pad")

handover_object_geom = mj_name2id(mjmodel, mujoco.mjtObj.mjOBJ_GEOM, "box_object")

In [ ]:
GEOM_IDX = mujoco.mjtObj.mjOBJ_GEOM
BODY_IDX = mujoco.mjtObj.mjOBJ_BODY
SITE_IDX = mujoco.mjtObj.mjOBJ_SITE
        
# Panda1 (left robot) indices
panda1_grip_site_idx = mj_name2id(mjmodel, SITE_IDX, "panda1_grip_site")
panda1_hand_body_idx = mj_name2id(mjmodel, BODY_IDX, "panda1_hand")
panda1_left_finger_geom = mj_name2id(mjmodel, GEOM_IDX, "panda1_leftfinger_collision1")
panda1_right_finger_geom = mj_name2id(mjmodel, GEOM_IDX, "panda1_rightfinger_collision1")

# Panda2 (right robot) indices
panda2_grip_site_idx = mj_name2id(mjmodel, SITE_IDX, "panda2_grip_site")
panda2_hand_body_idx = mj_name2id(mjmodel, BODY_IDX, "panda2_hand")
panda2_left_finger_geom = mj_name2id(mjmodel, GEOM_IDX, "panda2_leftfinger_collision1")
panda2_right_finger_geom = mj_name2id(mjmodel, GEOM_IDX, "panda2_rightfinger_collision1")

# Object indices
object_body_idx = mj_name2id(mjmodel, BODY_IDX, "handover_object")
object_geom_idx = mj_name2id(mjmodel, GEOM_IDX, "box_object")

# Goal location indices
handover_goal_idx = mj_name2id(mjmodel, SITE_IDX, "handover_goal")
place_goal_idx = mj_name2id(mjmodel, SITE_IDX, "place_goal")
pickup_goal_idx = mj_name2id(mjmodel, SITE_IDX, "pickup_goal")

# Table indices for place detection
table_right_geom = mj_name2id(mjmodel, GEOM_IDX, "table_right_top")

# Testing Full Env

In [ ]:
# Helpers for finding Contact IDs
def contact_id(pipeline_state: State, id1: int, id2: int) -> int:
    """Returns the contact id between two geom ids."""
    mask = (pipeline_state.contact.geom == jp.array([id1, id2])) | (pipeline_state.contact.geom == jp.array([id2, id1])) 
    mask2 = jp.all(mask[0], axis=1)
    id = jp.where(mask2)  # this was missing in the original code my bad
    return id

def all_contact_ids(pipeline_state: State, ids1: jp.ndarray, ids2: jp.ndarray) -> jp.ndarray:
    """Returns all contact ids between two sets of geom ids."""
    all_ids = []
    for id1 in ids1:
        for id2 in ids2:
            id = contact_id(pipeline_state, id1, id2)
            all_ids.append(id)
    return jp.concatenate(all_ids, axis=0)

In [ ]:
#from handover import CooperativeHandover
#env = CooperativeHandover()
from bedbathing import BedBathing 
env = BedBathing()

In [ ]:
N_ENVS = 1
rng = jax.random.PRNGKey(seed=0)
rng_env, rng_step = jax.random.split(rng, 2)
rng_envs = jax.random.split(rng_env, N_ENVS)
reset_jit = jax.jit(env.reset)
step_jit = jax.jit(env.step)
step_vmap = jax.vmap(step_jit, in_axes=(0,None))

state = jax.vmap(reset_jit)(rng_envs)

In [ ]:
def print_joint_info(env: PipelineEnv, state: State):
    # Verify joint indices
    print("=== JOINT INDICES ===")
    print(f"Total qpos size: {env.sys.q_size()}")
    print(f"Total qvel size: {env.sys.qd_size()}")
    print("\nqpos layout:")
    for i in range(env.sys.q_size()):
        print(f"  {i}: {state.pipeline_state.qpos[0, i]}")
    
    # Check joint names from MuJoCo model
    print("\n=== JOINT NAMES ===")
    from mujoco import mj_id2name, mjtObj
    mjmodel = env.sys.mj_model
    for i in range(mjmodel.njnt):
        name = mj_id2name(mjmodel, mjtObj.mjOBJ_JOINT, i)
        qpos_addr = mjmodel.jnt_qposadr[i]
        qvel_addr = mjmodel.jnt_dofadr[i]
        print(f"Joint {i}: '{name}' -> qpos[{qpos_addr}], qvel[{qvel_addr}]")



In [ ]:
print_joint_info(env, state)

In [ ]:
# In Jupyter notebook
def inspect_action_space(env):
    """Inspect action space of any environment."""
    mj_model = env.sys.mj_model
    n_actuators = mj_model.nu
    
    print(f"{'='*70}")
    print(f"ACTION SPACE INSPECTION")
    print(f"{'='*70}")
    print(f"Total actuators: {n_actuators}")
    print(f"Action size: {env.action_size}\n")
    
    # Group by robot
    panda1_acts = []
    panda2_acts = []
    
    print(f"{'Index':<8} {'Actuator Name':<30} {'Joint Name':<30}")
    print(f"{'-'*70}")
    
    for i in range(n_actuators):
        act_name = mj_model.actuator(i).name
        trnid = mj_model.actuator(i).trnid[0]
        
        if trnid >= 0:
            joint_name = mj_model.joint(trnid).name
        else:
            joint_name = "N/A"
        
        print(f"{i:<8} {act_name:<30} {joint_name:<30}")
        
        if 'panda1' in act_name:
            panda1_acts.append(i)
        elif 'panda2' in act_name:
            panda2_acts.append(i)
    
    print(f"\n{'='*70}")
    print(f"Robot 1 actuators: {panda1_acts} (count: {len(panda1_acts)})")
    print(f"Robot 2 actuators: {panda2_acts} (count: {len(panda2_acts)})")
    print(f"{'='*70}\n")
    
    # Suggest action ranges
    if panda1_acts and panda2_acts:
        print("Suggested action ranges:")
        print(f"  'robot1': [({min(panda1_acts)}, {max(panda1_acts)})],")
        print(f"  'robot2': [({min(panda2_acts)}, {max(panda2_acts)})],")
        print(f"  'global': [(0, {n_actuators - 1})],")

# Use it
inspect_action_space(env)

In [ ]:
# Verify joint indices
print("=== JOINT INDICES ===")
print(f"Total qpos size: {env.sys.q_size()}")
print(f"Total qvel size: {env.sys.qd_size()}")
print("\nqpos layout:")
for i in range(env.sys.q_size()):
    print(f"  {i}: {state.pipeline_state.qpos[0, i]}")

# Check joint names from MuJoCo model
print("\n=== JOINT NAMES ===")
from mujoco import mj_id2name, mjtObj
mjmodel = env.sys.mj_model
for i in range(mjmodel.njnt):
    name = mj_id2name(mjmodel, mjtObj.mjOBJ_JOINT, i)
    qpos_addr = mjmodel.jnt_qposadr[i]
    qvel_addr = mjmodel.jnt_dofadr[i]
    print(f"Joint {i}: '{name}' -> qpos[{qpos_addr}], qvel[{qvel_addr}]")

# Verify sensor indices
print("\n=== SENSOR INDICES ===")
print(f"Total sensors: {mjmodel.nsensor}")
for i in range(mjmodel.nsensor):
    name = mj_id2name(mjmodel, mjtObj.mjOBJ_SENSOR, i)
    print(f"Sensor {i}: '{name}' -> sensordata[{i}] = {state.pipeline_state.sensordata[0, i]}")

# Quick check of your indices
print("\n=== VERIFICATION ===")
print(f"Object joints (0-7): {state.pipeline_state.qpos[0, 0:7]}")
print(f"Panda1 joints (7-16): {state.pipeline_state.qpos[0, 7:16]}")
print(f"Panda2 joints (16-25): {state.pipeline_state.qpos[0, 16:25]}")

# VERIFY ENVIRONMENT DEFINITIONS
print("\n=== ENV DEFINITION CHECK ===")
print(f"env.object_joint_start = {env.object_joint_start} (expected: 0)")
print(f"env.object_joint_end = {env.object_joint_end} (expected: 7)")
print(f"env.panda1_joint_start = {env.panda1_joint_start} (expected: 7)")
print(f"env.panda1_joint_end = {env.panda1_joint_end} (expected: 16)")
print(f"env.panda2_joint_start = {env.panda2_joint_start} (expected: 16)")
print(f"env.panda2_joint_end = {env.panda2_joint_end} (expected: 25)")

print("\n=== SENSOR DEFINITION CHECK ===")
print(f"env.panda1_left_inner_touch_idx = {env.panda1_left_inner_touch_idx}")
print(f"env.panda1_right_inner_touch_idx = {env.panda1_right_inner_touch_idx}")
print(f"env.panda1_left_outer_touch_idx = {env.panda1_left_outer_touch_idx}")
print(f"env.panda1_right_outer_touch_idx = {env.panda1_right_outer_touch_idx}")
print(f"env.panda2_left_inner_touch_idx = {env.panda2_left_inner_touch_idx}")
print(f"env.panda2_right_inner_touch_idx = {env.panda2_right_inner_touch_idx}")
print(f"env.panda2_left_outer_touch_idx = {env.panda2_left_outer_touch_idx}")
print(f"env.panda2_right_outer_touch_idx = {env.panda2_right_outer_touch_idx}")

print("\n=== MATCH CHECK ===")
# Find actual indices from names and compare
for i in range(mjmodel.njnt):
    name = mj_id2name(mjmodel, mjtObj.mjOBJ_JOINT, i)
    qpos_addr = mjmodel.jnt_qposadr[i]
    if 'panda1' in name:
        if qpos_addr < env.panda1_joint_start or qpos_addr >= env.panda1_joint_end:
            print(f"❌ MISMATCH: {name} at qpos[{qpos_addr}] outside panda1 range [{env.panda1_joint_start}, {env.panda1_joint_end})")
    elif 'panda2' in name:
        if qpos_addr < env.panda2_joint_start or qpos_addr >= env.panda2_joint_end:
            print(f"❌ MISMATCH: {name} at qpos[{qpos_addr}] outside panda2 range [{env.panda2_joint_start}, {env.panda2_joint_end})")
    elif 'handover_object' in name:
        if qpos_addr < env.object_joint_start or qpos_addr >= env.object_joint_end:
            print(f"❌ MISMATCH: {name} at qpos[{qpos_addr}] outside object range [{env.object_joint_start}, {env.object_joint_end})")

for i in range(mjmodel.nsensor):
    name = mj_id2name(mjmodel, mjtObj.mjOBJ_SENSOR, i)
    if name == 'panda1_left_inner_touch' and i != env.panda1_left_inner_touch_idx:
        print(f"❌ MISMATCH: {name} is sensor {i} but env has {env.panda1_left_inner_touch_idx}")
    elif name == 'panda1_right_inner_touch' and i != env.panda1_right_inner_touch_idx:
        print(f"❌ MISMATCH: {name} is sensor {i} but env has {env.panda1_right_inner_touch_idx}")
    # Add more sensor checks...

print("\n✓ If no mismatches printed above, all indices are correct!")

In [ ]:
rng = jax.random.key(0)
rng, rng_step = jax.random.split(rng)
print(rng_step)
random_action = jax.random.uniform(rng_step, (env.action_size,), minval=-1.0, maxval=1.0)
next_state = step_vmap(rng_step, state, random_action)

In [ ]:
next_state.reward

In [ ]:
next_state.obs.shape

In [ ]:
panda1_left_finger_geom = env.panda1_left_finger_geom
panda1_right_finger_geom = env.panda1_right_finger_geom
panda2_left_finger_geom = env.panda2_left_finger_geom
panda2_right_finger_geom = env.panda2_right_finger_geom
handover_object_geom = env.object_geom_idx

In [ ]:
print(f"""
Panda 1 Left Finger Geometry ID: {panda1_left_finger_geom}
Panda 1 Right Finger Geometry ID: {panda1_right_finger_geom}
Panda 2 Left Finger Geometry ID: {panda2_left_finger_geom}
Panda 2 Right Finger Geometry ID: {panda2_right_finger_geom}
Handover Object Geometry ID: {handover_object_geom}
""")

In [ ]:
def contact_id(pipeline_state: State, id1: int, id2: int) -> int:
    """Returns the contact id between two geom ids."""
    mask = (pipeline_state.contact.geom == jp.array([id1, id2])) | (pipeline_state.contact.geom == jp.array([id2, id1])) 
    mask2 = jp.all(mask[0], axis=1)
    id = jp.where(mask2)  # this was missing in the original code my bad
    return id

In [ ]:
panda_wiper = env.panda_wiper_idx
human_uarm = env.human_tuarm_geom
human_larm = env.human_tlarm_geom

In [ ]:
contact_wiper_uarm = contact_id(state.pipeline_state, panda_wiper, human_uarm)
contact_wiper_larm = contact_id(state.pipeline_state, panda_wiper, human_larm)
print(f"Contact IDs between wiper and human upper arm: {contact_wiper_uarm}")
print(f"Contact IDs between wiper and human lower arm: {contact_wiper_larm}")

In [ ]:
contact_panda1_left = contact_id(state.pipeline_state, panda1_left_finger_geom, handover_object_geom)
contact_panda1_right = contact_id(state.pipeline_state, panda1_right_finger_geom, handover_object_geom)
contact_panda2_left = contact_id(state.pipeline_state, panda2_left_finger_geom, handover_object_geom)
contact_panda2_right = contact_id(state.pipeline_state, panda2_right_finger_geom, handover_object_geom)

print("Contact IDs:")
print(f"  Panda 1 Left Finger - Handover Object: {contact_panda1_left}")
print(f"  Panda 1 Right Finger - Handover Object: {contact_panda1_right}")
print(f"  Panda 2 Left Finger - Handover Object: {contact_panda2_left}")
print(f"  Panda 2 Right Finger - Handover Object: {contact_panda2_right}")

In [ ]:
int(contact_panda1_left[0][0])

In [ ]:
contact_panda1_left_forces1 = contact_force(env.sys, state.pipeline_state, int(contact_panda1_left[0][0]), False)
contact_panda1_left_forces2 = contact_force(env.sys, state.pipeline_state, int(contact_panda1_left[0][1]))
contact_panda1_left_forces3 = contact_force(env.sys, state.pipeline_state, int(contact_panda1_left[0][2]))
contact_panda1_left_forces4 = contact_force(env.sys, state.pipeline_state, int(contact_panda1_left[0][3]))

contact_panda1_right_forces1 = contact_force(env.sys, state.pipeline_state, int(contact_panda1_right[0][0]))
contact_panda1_right_forces2 = contact_force(env.sys, state.pipeline_state, int(contact_panda1_right[0][1]))
contact_panda1_right_forces3 = contact_force(env.sys, state.pipeline_state, int(contact_panda1_right[0][2]))
contact_panda1_right_forces4 = contact_force(env.sys, state.pipeline_state, int(contact_panda1_right[0][3]))

print(f"""
Panda 1 Left Finger - Handover Object Contact Forces:
  Contact 1 Forces: {contact_panda1_left_forces1}
  Contact 2 Forces: {contact_panda1_left_forces2}
  Contact 3 Forces: {contact_panda1_left_forces3}
  Contact 4 Forces: {contact_panda1_left_forces4}
Panda 1 Right Finger - Handover Object Contact Forces:
    Contact 1 Forces: {contact_panda1_right_forces1}
    Contact 2 Forces: {contact_panda1_right_forces2}
    Contact 3 Forces: {contact_panda1_right_forces3}
    Contact 4 Forces: {contact_panda1_right_forces4}
""")

In [ ]:
state.pipeline_state.pipeline_state

In [ ]:
floor = env.floor_geom_idx
handover_object = env.object_geom_idx

floor_handover_contact_id = contact_id(state.pipeline_state, floor, handover_object)   
object_floor_contact_id = contact_id(state.pipeline_state, handover_object, floor)
print(f"floor_handover_contact_id: {floor_handover_contact_id} \n object_floor_contact_id: {object_floor_contact_id}")

In [ ]:
x = int(contact[0][3])

In [ ]:
x

In [ ]:
next_state.pipeline_state

In [ ]:
state

In [ ]:
contact_forces = contact_force(env.sys, state.pipeline_state, x, False)

In [ ]:
efc_adress = next_state.pipeline_state.contact.efc_address[x]

In [ ]:
efc_adress

In [ ]:
next_state.pipeline_state._impl.efc_force[0][efc_adress:]

In [ ]:
next_state.pipeline_state._impl.efc_force.shape

In [ ]:
condim = next_state.pipeline_state._impl.contact.dim[x]

In [ ]:
condim

# Random PushCoop Rendering Debug

In [ ]:
import safetensors.flax

def _tree_shape(pytree):
    return jax.tree.map(lambda x: x.shape, pytree)

def _tree_take(pytree, indices, axis=None):
    return jax.tree.map(lambda x: x.take(indices, axis=axis), pytree)



In [ ]:
robot_params_path = '/home/leo/assistive-autonomy-github/assistax/outputs/IPPO/scratchitch/2025-10-20/18-04-15/robot.safetensors'
human_params_path = '/home/leo/assistive-autonomy-github/assistax/outputs/IPPO/scratchitch/2025-10-20/18-04-15/human.safetensors' 
all_params_path = '/home/leo/assistive-autonomy-github/assistax/outputs/IPPO/scratchitch/2025-10-20/18-04-15/all_params.safetensors'
pushcoop_ps_path = '/home/leo/assistive-autonomy-github/assistax/outputs/pushcoop/IPPO/2025-10-25/15-44-41/final_params.safetensors'

In [ ]:
robo_params_shape = _tree_shape(safetensors.flax.load_file(robot_params_path))
human_params_shape = _tree_shape(safetensors.flax.load_file(human_params_path))
all_params_shape = _tree_shape(safetensors.flax.load_file(all_params_path))
pushcoop_ps_shape = _tree_shape(safetensors.flax.load_file(pushcoop_ps_path))

print("Robot Params Shape:\n", robo_params_shape)
print("\nHuman Params Shape:\n", human_params_shape)
print("\nAll Params Shape:\n", all_params_shape)
print("\nPushCoop Params Shape:\n", pushcoop_ps_shape)

In [ ]:
test = _tree_take(safetensors.flax.load_file(pushcoop_ps_path), 1, axis=0)

In [ ]:
_tree_shape(test)

# Feeding Task

In [ ]:
from typing import Tuple
from enum import IntEnum

from brax import base
from brax.envs.base import PipelineEnv, State
from brax.io import mjcf
from etils import epath
import jax
from jax import numpy as jp
import mujoco 
from mujoco import mj_id2name, mj_name2id
from mujoco.mjx._src.support import contact_force # this import causes the warp warning due to some imports in that file idk why or if important

In [ ]:
#from handover import CooperativeHandover
#env = CooperativeHandover()
from feeding import Feeding 
env = Feeding()

In [ ]:
# Helpers for finding Contact IDs
def contact_id(pipeline_state: State, id1: int, id2: int) -> int:
    """Returns the contact id between two geom ids."""
    mask = (pipeline_state.contact.geom == jp.array([id1, id2])) | (pipeline_state.contact.geom == jp.array([id2, id1])) 
    mask2 = jp.all(mask[0], axis=1)
    id = jp.where(mask2)  # this was missing in the original code my bad
    return id

def all_contact_ids(pipeline_state: State, ids1: jp.ndarray, ids2: jp.ndarray) -> jp.ndarray:
    """Returns all contact ids between two sets of geom ids."""
    all_ids = []
    for id1 in ids1:
        for id2 in ids2:
            id = contact_id(pipeline_state, id1, id2)
            all_ids.append(id)
    return jp.concatenate(all_ids, axis=0)

In [ ]:
N_ENVS = 1
rng = jax.random.PRNGKey(seed=0)
rng_env, rng_step = jax.random.split(rng, 2)
rng_envs = jax.random.split(rng_env, N_ENVS)
reset_jit = jax.jit(env.reset)
step_jit = jax.jit(env.step)
step_vmap = jax.vmap(step_jit, in_axes=(0,None))

state = jax.vmap(reset_jit)(rng_envs)

In [ ]:
def contact_id(pipeline_state: State, id1: int, id2: int) -> int:
    """Returns the contact id between two geom ids."""
    mask = (pipeline_state.contact.geom == jp.array([id1, id2])) | (pipeline_state.contact.geom == jp.array([id2, id1])) 
    mask2 = jp.all(mask[0], axis=1)
    id = jp.where(mask2)  # this was missing in the original code my bad
    return id

In [ ]:
spoon = env.panda_spoon_geom_idx
spoon_rside = env.panda_spoon_rside_geom_idx
h_head = env.human_head_geom_idx 

spoon_head_contact_id = contact_id(state.pipeline_state, spoon, h_head)
spoon_rside_head_contact_id = contact_id(state.pipeline_state, spoon_rside, h_head)

print(f"Spoon Head Contact: {spoon_head_contact_id} \
      Spoon R Side Head Contact: {spoon_rside_head_contact_id}")


In [ ]:
# Verify joint indices
print("=== JOINT INDICES ===")
print(f"Total qpos size: {env.sys.q_size()}")
print(f"Total qvel size: {env.sys.qd_size()}")
print("\nqpos layout:")
for i in range(env.sys.q_size()):
    print(f"  {i}: {state.pipeline_state.qpos[0, i]}")

# Check joint names from MuJoCo model
print("\n=== JOINT NAMES ===")
from mujoco import mj_id2name, mjtObj
mjmodel = env.sys.mj_model
for i in range(mjmodel.njnt):
    name = mj_id2name(mjmodel, mjtObj.mjOBJ_JOINT, i)
    qpos_addr = mjmodel.jnt_qposadr[i]
    qvel_addr = mjmodel.jnt_dofadr[i]
    print(f"Joint {i}: '{name}' -> qpos[{qpos_addr}], qvel[{qvel_addr}]")

# Verify sensor indices
print("\n=== SENSOR INDICES ===")
print(f"Total sensors: {mjmodel.nsensor}")
for i in range(mjmodel.nsensor):
    name = mj_id2name(mjmodel, mjtObj.mjOBJ_SENSOR, i)
    print(f"Sensor {i}: '{name}' -> sensordata[{i}] = {state.pipeline_state.sensordata[0, i]}")

# Quick check of your indices
print("\n=== VERIFICATION ===")
print(f"Object joints (0-7): {state.pipeline_state.qpos[0, 0:7]}")
print(f"Panda1 joints (7-16): {state.pipeline_state.qpos[0, 7:16]}")
print(f"Panda2 joints (16-25): {state.pipeline_state.qpos[0, 16:25]}")

# VERIFY ENVIRONMENT DEFINITIONS
print("\n=== ENV DEFINITION CHECK ===")
print(f"env.panda_joint_start = {env.panda_joint_start} (expected: 7)")
print(f"env.panda_joint_end = {env.panda_joint_end} (expected: 16)")
print(f"env.human_joint_start = {env.human_joint_start} (expected: 16)")
print(f"env.human_joint_end = {env.human_joint_end} (expected: 25)")


print("\n=== MATCH CHECK ===")
# Find actual indices from names and compare
for i in range(mjmodel.njnt):
    name = mj_id2name(mjmodel, mjtObj.mjOBJ_JOINT, i)
    qpos_addr = mjmodel.jnt_qposadr[i]
    if 'panda1' in name:
        if qpos_addr < env.panda_joint_id_start or qpos_addr >= env.panda_joint_id_end:
            print(f"❌ MISMATCH: {name} at qpos[{qpos_addr}] outside panda range [{env.panda_joint_start}, {env.panda_joint_end})")
    elif 'panda2' in name:
        if qpos_addr < env.human_joint_id_start or qpos_addr >= env.human_joint_id_end:
            print(f"❌ MISMATCH: {name} at qpos[{qpos_addr}] outside human range [{env.human_joint_start}, {env.human_joint_end})")
    
print("\n✓ If no mismatches printed above, all indices are correct!")

# Teethbrushing

In [1]:
from typing import Tuple
from enum import IntEnum

from brax import base
from brax.envs.base import PipelineEnv, State
from brax.io import mjcf
from etils import epath
import jax
from jax import numpy as jp
import mujoco 
from mujoco import mj_id2name, mj_name2id
from mujoco.mjx._src.support import contact_force # this import causes the warp warning due to some imports in that file idk why or if important

Failed to import warp: No module named 'warp'
Failed to import mujoco_warp: No module named 'warp'


In [2]:
#from handover import CooperativeHandover
#env = CooperativeHandover()
from teethbrushing import TeethBrushing 
env = TeethBrushing()

/home/leo/assistive-autonomy-github/assistax/.venv/lib/python3.11/site-packages/brax/io/mjcf.py:480: UserWarning: Brax System, piplines and environments are not actively being maintained. Please see MJX for a well maintained JAX-based physics engine: https://github.com/google-deepmind/mujoco/tree/main/mjx. For a host of environments that use MJX, see: https://github.com/google-deepmind/mujoco_playground.
  warnings.warn(


In [3]:
N_ENVS = 1
rng = jax.random.PRNGKey(seed=0)
rng_env, rng_step = jax.random.split(rng, 2)
rng_envs = jax.random.split(rng_env, N_ENVS)
reset_jit = jax.jit(env.reset)
step_jit = jax.jit(env.step)
step_vmap = jax.vmap(step_jit, in_axes=(0,None))

state = jax.vmap(reset_jit)(rng_envs)

/home/leo/assistive-autonomy-github/assistax/.venv/lib/python3.11/site-packages/jax/_src/abstract_arrays.py:135: RuntimeWarning: overflow encountered in cast
  return literals.TypedNdArray(np.asarray(x, dtype), weak_type=False)


In [4]:
def contact_id(pipeline_state: State, id1: int, id2: int) -> int:
    """Returns the contact id between two geom ids."""
    mask = (pipeline_state.contact.geom == jp.array([id1, id2])) | (pipeline_state.contact.geom == jp.array([id2, id1])) 
    mask2 = jp.all(mask[0], axis=1)
    id = jp.where(mask2)  # this was missing in the original code my bad
    return id

In [5]:
toothbrush = env.panda_toothbrush_geom_idx
toothbrush_rside = env.panda_toothbrush_rside_geom_idx
h_head = env.human_head_geom_idx 

toothbrush_head_contact_id = contact_id(state.pipeline_state, toothbrush, h_head)
toothbrush_rside_head_contact_id = contact_id(state.pipeline_state, toothbrush_rside, h_head)

print(f"Toothbrush Head Contact: {toothbrush_head_contact_id} \
      Toothbrush R Side Head Contact: {toothbrush_rside_head_contact_id}")

Toothbrush Head Contact: (Array([19], dtype=int32),)       Toothbrush R Side Head Contact: (Array([20], dtype=int32),)


# Bed bathing ncon check

In [2]:
from bedbathing import BedBathing 
env = BedBathing()

/home/leo/assistive-autonomy-github/assistax/.venv/lib/python3.11/site-packages/brax/io/mjcf.py:480: UserWarning: Brax System, piplines and environments are not actively being maintained. Please see MJX for a well maintained JAX-based physics engine: https://github.com/google-deepmind/mujoco/tree/main/mjx. For a host of environments that use MJX, see: https://github.com/google-deepmind/mujoco_playground.
  warnings.warn(


In [3]:
N_ENVS = 1
rng = jax.random.PRNGKey(seed=0)
rng_env, rng_step = jax.random.split(rng, 2)
rng_envs = jax.random.split(rng_env, N_ENVS)
reset_jit = jax.jit(env.reset)
step_jit = jax.jit(env.step)
step_vmap = jax.vmap(step_jit, in_axes=(0,None))

state = jax.vmap(reset_jit)(rng_envs)

/home/leo/assistive-autonomy-github/assistax/.venv/lib/python3.11/site-packages/jax/_src/abstract_arrays.py:135: RuntimeWarning: overflow encountered in cast
  return literals.TypedNdArray(np.asarray(x, dtype), weak_type=False)


In [4]:
state.pipeline_state.ncon

/tmp/ipykernel_93765/2644446879.py:1: DeprecationWarning: Accessing `ncon` directly from `Data` is deprecated. Access it via `data._impl.ncon` instead.
  state.pipeline_state.ncon


280

In [5]:
from scratchitch import ScratchItch
env = ScratchItch()

N_ENVS = 1
rng = jax.random.PRNGKey(seed=0)
rng_env, rng_step = jax.random.split(rng, 2)
rng_envs = jax.random.split(rng_env, N_ENVS)
reset_jit = jax.jit(env.reset)
step_jit = jax.jit(env.step)
step_vmap = jax.vmap(step_jit, in_axes=(0,None))
state = jax.vmap(reset_jit)(rng_envs)

/home/leo/assistive-autonomy-github/assistax/.venv/lib/python3.11/site-packages/brax/io/mjcf.py:480: UserWarning: Brax System, piplines and environments are not actively being maintained. Please see MJX for a well maintained JAX-based physics engine: https://github.com/google-deepmind/mujoco/tree/main/mjx. For a host of environments that use MJX, see: https://github.com/google-deepmind/mujoco_playground.
  warnings.warn(
/home/leo/assistive-autonomy-github/assistax/.venv/lib/python3.11/site-packages/jax/_src/abstract_arrays.py:135: RuntimeWarning: overflow encountered in cast
  return literals.TypedNdArray(np.asarray(x, dtype), weak_type=False)


In [6]:
state.pipeline_state.ncon

/tmp/ipykernel_93765/2644446879.py:1: DeprecationWarning: Accessing `ncon` directly from `Data` is deprecated. Access it via `data._impl.ncon` instead.
  state.pipeline_state.ncon


282